# Agentic RAG

## Installing Packages

In [1]:
#pip install langchain-groq faiss-cpu crewai serper pypdf2 python-dotenv setuptools sentence-transformers huggingface distutils

In [2]:
pip install langchain-groq faiss-cpu "crewai[tools]" pypdf2 python-dotenv sentence-transformers huggingface-hub


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setting Up the Environment

In [3]:
import os

from dotenv import load_dotenv

#from langchain.vectorstores import FAISS
from langchain_community.vectorstores import FAISS

#from langchain.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

#from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

#from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_groq import ChatGroq

from crewai_tools import SerperDevTool

from crewai import Agent, Task, Crew, LLM

load_dotenv()

# API Keys
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
GEMINI = os.getenv("GEMINI")

/Users/suraj/Machine Learning/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Initializing the LLMs

In [4]:
import os
print(f"Current Directory: {os.getcwd()}")
print(f"Files in this folder: {os.listdir('.')}")

Current Directory: /Users/suraj/Machine Learning/GenAI
Files in this folder: ['agentic_rag.ipynb', '.DS_Store', 'Agentic_RAG.py', '.env', 'What is an AI agent.pdf']


In [5]:
# # Run this cell once to create your .env file
# env_content = """
# GROQ_API_KEY=your_actual_groq_key_here
# SERPER_API_KEY=your_actual_serper_key_here
# GEMINI=your_actual_gemini_key_here
# """

# with open(".env", "w") as f:
#     f.write(env_content.strip())

# print("✅ .env file created successfully!")

In [6]:
# import os
# from dotenv import load_dotenv, find_dotenv

# # Automatically finds the .env file in the current or parent directories
# load_dotenv(find_dotenv(), override=True)

# GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# GEMINI = os.getenv("GEMINI")

# if GROQ_API_KEY:
#     print(f"✅ Success! Key starts with: {GROQ_API_KEY[:5]}...")
# else:
#     print("❌ Error: Key still not found. Check if the file was created.")

In [7]:
llm = ChatGroq(
model="llama-3.3-70b-specdec",
temperature=0,
max_tokens=500,
timeout=None,
max_retries=2,
)
crew_llm = LLM(
model="gemini/gemini-1.5-flash",
api_key=GEMINI,
max_tokens=500,
temperature=0.7
)

## Decision Maker (Router)

In [8]:
# def check_local_knowledge (query, context):
# prompt = '''Role: Question-Answering Assistant
# Task: Determine whether the system can answer the user's question based on the provided text.
# Output Format: Answer: Yes/No
# User Question: {query}
# Text: {text}
# formatted_prompt = prompt.format(text=context,
# query=query)
# response = llm.invoke(formatted_prompt)
# return response.content.strip().lower() = "yes"

In [9]:
def check_local_knowledge(query, context):
    prompt = '''Role: Question-Answering Assistant
Task: Determine whether the system can answer the user's question based on the provided text.
Output Format: Answer: Yes/No
User Question: {query}
Text: {text}'''
    
    formatted_prompt = prompt.format(text=context, query=query)
    response = llm.invoke(formatted_prompt)
    
    return response.content.strip().lower().startswith("yes")

## Web Searching and Scraping Agent

In [10]:
def setup_web_scraping_agent():
    search_tool = SerperDevTool()
    scrape_website = ScrapeWebsiteTool()
    web_search_agent = Agent(role="Expert Web Search Agent",goal="Identify and retrieve relevant web data for user queries",llm=crew_llm)
    web_scraper_agent = Agent(role="Expert Web Scraper Agent",goal="Extract and analyze content from webpages",llm=crew_llm)
    
    return crew

def get_web_content(query):
    crew = setup_web_scraping_agent()
    result = crew.kickoff(inputs={"topic": query})
    
    return result.raw

## Creating the Vector Database

In [11]:
def setup_vector_db(pdf_path):
    loader = PyPDFLoader (pdf_path)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=50)
    chunks = text_splitter.split_documents (documents)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    vector_db = FAISS.from_documents(chunks,embeddings)
    return vector_db

def get_local_content(vector_db, query):
    docs = vector_db.similarity_search(query, k = 5 )
    return " ".join([doc.page_content for doc in docs])

## Generating the Final Answer

In [12]:
def generate_final_answer (context, query):
    messages = [
    ("system", "You are a helpful assistant. Use the provided context to answer the query accurately."),
    ("system", f"Context: {context}"),
    ("human", query),
    ]
    response = llm.invoke(messages)
    return response.content

def process_query(query, vector_db, local_context):
    can_answer_locally = check_local_knowledge (query,local_context)
    context = get_local_content(vector_db, query) if can_answer_locally else get_web_content(query)
    return generate_final_answer(context, query)

## Main: Tie Everything Together

In [13]:
def main():
    pdf_path = "genai-principles.pdf"
    vector_db = setup_vector_db(pdf_path)
    local_context = get_local_content(vector_db, "")
    query = "What is Agentic RAG?"
    result = process_query(query, vector_db,
    local_context)
    print("\nFinal Answer:")
    print(result)

#if_name = "_main__":
    #main()